# Equações de Friedmann

A métrica de Friedmann–Lemaître–Robertson–Walker (FLRW) descreve um universo homogêneo e isotrópico:

$$ds^2=-c^2dt^2+a(t)^2\left[\frac{dr^2}{1-kr^2}+r^2d\Omega^2\right],\quad k=0,\pm1.$$

onde 

$$d\Omega^2 = d \theta² + \sin²\theta d \phi²$$

Usaremos a equação de Einstein adotada acima, $G_{\mu\nu}+\Lambda g_{\mu\nu}=(8\pi G/c^4)T_{\mu\nu}$, e a convenção de Ricci já implementada neste notebook. A fonte é um fluido perfeito com densidade de energia $\varepsilon$ e pressão $p$, no referencial comóvel: $T_{00}=\varepsilon$ e $T_{ij}=p g_{ij}$. Com $H=\dot a/a$, as componentes geométricas são

$$G_{00}=3\left(H^2+\frac{kc^2}{a^2}\right),\quad G_{ij}=-\frac{1}{c^2}\left(2\frac{\ddot a}{a}+H^2+\frac{kc^2}{a^2}\right)g_{ij}.$$

A equação $00$ e uma equação espacial fornecem as duas equações de Friedmann. No caso $k=0$, $\Lambda=0$, a primeira reduz-se a $H^2=8\pi G\varepsilon/(3c^2)$; se $\varepsilon=\rho c^2$, recuperamos $H^2=8\pi G\rho/3$.

# Termos Geométricos

In [2]:
# METADADOS & PACOTES
from sympy import *
from itertools import *

import matplotlib.pyplot as plt
import numpy as np
# Einsteinpy: https://docs.einsteinpy.org/en/stable/index.html
# Sympy: https://docs.sympy.org/latest/index.html
# Intertools: https://docs.python.org/3/library/itertools.html

In [11]:
# PARÂMETROS 
# Curvatura gaussiana k, Constante de Newton G, velocidade da luz c, constante cosmológica Lb, tempo t, raio r, angulo polar theta th, azimuth phi ph 
k, G, c, Lb, t, r, th, ph = symbols('k G c Lb t r th ph')
X = [t, r, th, ph]
# Também definimos a função a(t) que é a função a ser determinanda pela equação de Einstein
a = Function('a')('t') 

## Métrica

In [12]:
# Aqui vamos definir a métrica. Inicialmente preenchemos todas as entradas com zeros, e depois definimos as componentes não nulas. 
g = zeros(4) 
g[0,0] = -c**2 # felizmente o python é relativista e começa a contar os indices a partir do zero 
g[1,1] = a / (1 - k* (r**2))
g[2,2] = r**2
g[3,3] = r**2 * sin(th)**2

rank = g.rank() # o rank será útil para generalizar esse código para casos onde o espaço-tempo tem dimensões maiores ou menos que 4. Sim, isso é muito importante. 
gI = g.inv() # Também precisamos da inversa para subir os indices de Lorentz

## Símbolos de Christoffel

Esses objetos são as componentes da conexão de Levi-Civita dadas explicitamente por:

\begin{equation}
\Gamma^\rho_{\mu\nu} = \frac{1}{2} g^{\rho\sigma} \left( \partial_\mu g_{\sigma\nu} + \partial_\sigma g_{\mu\nu} - \partial_\sigma g_{\mu\nu} \right)
\end{equation}

In [13]:
# k denota os indices contravariantes (no andar de cima), ao passo que i e j denotam os indices covariantes (no andar de baixo).
def Gamma(k,i,j):
    n = 0
    GammaMatrix = 0
    while n < rank:
        GammaMatrix = GammaMatrix + gI[k,n]*(g[n,j].diff(X[i]) + g[n,i].diff(X[j]) - g[i,j].diff(X[n])) / 2
        n += 1
    return GammaMatrix

É útil pensar nesses objetos, os símbolos de Christoffel como 4 matrizes rotulados com o indice k = 0, 1, 2, 3, de tal forma que:

In [14]:
# Primeiro construímos 4 matrizes com todas as componentes nulas
ChSymb0 = zeros(rank)
ChSymb1 = zeros(rank)
ChSymb2 = zeros(rank)
ChSymb3 = zeros(rank)
ChSymb = [ChSymb0, ChSymb1, ChSymb2, ChSymb3]

In [15]:
# Agora preenchemos essas matrizes com as componentes não nulas para o ansätz dado em (3)
for k in range(rank):
    for i in range(rank):
        for j in range(rank):
            if Gamma(k,i,j) != 0:
                ChSymb[k][i,j] = Gamma(k,i,j)

In [16]:
# Para ver o resultado explícito, fazemos
for k in range(len(ChSymb)):
    print(f"+++++++++ Christoffel({k}) ++++++++++++++++")
    display(ChSymb[k])
    print(" ")

+++++++++ Christoffel(0) ++++++++++++++++


Matrix([
[0,                                          0, 0, 0],
[0, Derivative(a(t), t)/(2*c**2*(-k*r**2 + 1)), 0, 0],
[0,                                          0, 0, 0],
[0,                                          0, 0, 0]])

 
+++++++++ Christoffel(1) ++++++++++++++++


Matrix([
[                           0, Derivative(a(t), t)/(2*a(t)),                     0,                                0],
[Derivative(a(t), t)/(2*a(t)),            k*r/(-k*r**2 + 1),                     0,                                0],
[                           0,                            0, -r*(-k*r**2 + 1)/a(t),                                0],
[                           0,                            0,                     0, -r*(-k*r**2 + 1)*sin(th)**2/a(t)]])

 
+++++++++ Christoffel(2) ++++++++++++++++


Matrix([
[0,   0,   0,                0],
[0,   0, 1/r,                0],
[0, 1/r,   0,                0],
[0,   0,   0, -sin(th)*cos(th)]])

 
+++++++++ Christoffel(3) ++++++++++++++++


Matrix([
[0,   0,               0,               0],
[0,   0,               0,             1/r],
[0,   0,               0, cos(th)/sin(th)],
[0, 1/r, cos(th)/sin(th),               0]])

## Tensor de Riemann

De posse dos simbolos de Christoffel, podemos encontrar o tensor de Riemann pela expressão

\begin{equation}
R^{\lambda}_{\rho \mu \nu} = \partial_\mu \Gamma^{\lambda}_{\nu \rho} - \partial_\lambda \Gamma^\nu_{\rho \mu} + \Gamma^\lambda_{\mu\sigma}\Gamma^\sigma_{\rho\nu} -  \Gamma^\lambda_{\nu \sigma}\Gamma^\sigma_{\rho\mu}
\end{equation}

In [17]:
# l é contravariante; k,i,j são covariantes
# Aqui vamos definir a constração do produto de dois simbolos de Christoffel. Ou seja, os dois últimos termos na equação (5)
def Contraction(l,k,i,j):
    n = 0 
    cont = 0 
    while n < rank:
        cont = cont + ChSymb[l][i,n] * ChSymb[n][k,j] - ChSymb[l][j,n] * ChSymb[n][k,i]
        n += 1
    return cont

In [18]:
# Agora calculamos as componentes do tensor de Riemann propriamente ditas. 
def Riem(l,k,i,j):
    n = 0
    Riemann = 0
    while n < rank:
        Riemann = Riemann + ChSymb[l][k,j].diff(X[i]) - ChSymb[l][k,i].diff(X[j]) + Contraction(l,k,i,j)
        n += 1
    R = Riemann.simplify()
    return R

Com as componentes que podemos calcular usando a função acima, vamos organizar esses resultados em termos de matrizes $(R^\lambda_\rho)_{\mu\nu}$

In [19]:
# Mais uma vez, definimos as matrizes e preenchemos com zeros. 
RieTen00 = zeros(rank); RieTen01 = zeros(rank); RieTen02 = zeros(rank); RieTen03 = zeros(rank)
RieTen10 = zeros(rank); RieTen11 = zeros(rank); RieTen12 = zeros(rank); RieTen13 = zeros(rank)
RieTen20 = zeros(rank); RieTen21 = zeros(rank); RieTen22 = zeros(rank); RieTen23 = zeros(rank)
RieTen30 = zeros(rank); RieTen31 = zeros(rank); RieTen32 = zeros(rank); RieTen33 = zeros(rank)

Rie = [[RieTen00, RieTen01, RieTen02, RieTen03], 
       [RieTen10, RieTen11, RieTen12, RieTen13],
       [RieTen20, RieTen21, RieTen22, RieTen23],
       [RieTen30, RieTen31, RieTen32, RieTen33]]

In [20]:
# E agora podemos preencher as componentes não nulas nas matrizes acima
for l in range(rank):
    for k in range(rank):
        for i in range(rank):
            for j in range(rank):
                if Riem(l,k,i,j) != 0:
                    Rie[l][k][i,j] = Riem(l,k,i,j) / 4 
                    '''
                    Aqui temos que explicar a presença dessa divisão por 4. No meu código, estamos separando um tensor em várias matrizes, 
                    e isso introduz redundâncias devido aos loops que estamos tomando sobre esses objetos. 
                    '''

In [21]:
# Para ver o resultado explícito, fazemos
for l in range(rank):
    for k in range(rank):
        display(Rie[l][k])

Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])

Matrix([
[                                                                                    0, (-2*a(t)*Derivative(a(t), (t, 2)) + Derivative(a(t), t)**2)/(4*c**2*(k*r**2 - 1)*a(t)), 0, 0],
[(2*a(t)*Derivative(a(t), (t, 2)) - Derivative(a(t), t)**2)/(4*c**2*(k*r**2 - 1)*a(t)),                                                                                      0, 0, 0],
[                                                                                    0,                                                                                      0, 0, 0],
[                                                                                    0,                                                                                      0, 0, 0]])

Matrix([
[0,                                   0,                                    0, 0],
[0,                                   0, -r*Derivative(a(t), t)/(2*c**2*a(t)), 0],
[0, r*Derivative(a(t), t)/(2*c**2*a(t)),                                    0, 0],
[0,                                   0,                                    0, 0]])

Matrix([
[0,                                              0, 0,                                               0],
[0,                                              0, 0, -r*sin(th)**2*Derivative(a(t), t)/(2*c**2*a(t))],
[0,                                              0, 0,                                               0],
[0, r*sin(th)**2*Derivative(a(t), t)/(2*c**2*a(t)), 0,                                               0]])

Matrix([
[                                                                      0, (2*a(t)*Derivative(a(t), (t, 2)) - Derivative(a(t), t)**2)/(4*a(t)**2), 0, 0],
[(-2*a(t)*Derivative(a(t), (t, 2)) + Derivative(a(t), t)**2)/(4*a(t)**2),                                                                      0, 0, 0],
[                                                                      0,                                                                      0, 0, 0],
[                                                                      0,                                                                      0, 0, 0]])

...

Matrix([
[0, 0,                                    0,                                   0],
[0, 0,                                    0,                                   0],
[0, 0,                                    0, (k*r**2 + a(t) - 1)*sin(th)**2/a(t)],
[0, 0, (-k*r**2 - a(t) + 1)*sin(th)**2/a(t),                                   0]])

Matrix([
[0,                              0, 0,                               0],
[0,                              0, 0, -Derivative(a(t), t)/(2*r*a(t))],
[0,                              0, 0,                               0],
[0, Derivative(a(t), t)/(2*r*a(t)), 0,                               0]])

Matrix([
[                             0,               0, 0, -Derivative(a(t), t)/(2*r*a(t))],
[                             0,               0, 0,                  k/(k*r**2 - 1)],
[                             0,               0, 0,                               0],
[Derivative(a(t), t)/(2*r*a(t)), -k/(k*r**2 - 1), 0,                               0]])

Matrix([
[0, 0,                        0,                         0],
[0, 0,                        0,                         0],
[0, 0,                        0, (-k*r**2 - a(t) + 1)/a(t)],
[0, 0, (k*r**2 + a(t) - 1)/a(t),                         0]])

Matrix([
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]])

## Tensor de Ricci

Finalmente, podemos calcular o tensor de Ricci como 
\begin{equation}
R_{\mu\nu} = R^\rho_{\mu \rho \nu}
\end{equation}

In [22]:
def Ricci(i,j):
    ricc = 0
    n = 0
    for k in range(rank):
        ricc = ricc + Rie[k][i][k,j]
        r = ricc.simplify()
    return r

Agora vamos organizar esse tensor na forma matricial 

In [23]:
Ric = zeros(rank)

In [24]:
# E agora podemos preencher as componentes não nulas nas matrizes acima
for i in range(rank):
    for j in range(rank):
         if Ricci(i,j) != 0:
            Ric[i,j] = Ricci(i,j)

In [25]:
# Para ver o resultado explícito, fazemos
for i in range(rank):
    for j in range(rank):
        if Ric[i,j] != 0:
            print(f"+++++++++ Ricci({i},{j}) ++++++++++++++++")
            display(Ric[i,j])
            print(" ")

+++++++++ Ricci(0,0) ++++++++++++++++


(-2*a(t)*Derivative(a(t), (t, 2)) + Derivative(a(t), t)**2)/(4*a(t)**2)

 
+++++++++ Ricci(0,1) ++++++++++++++++


Derivative(a(t), t)/(r*a(t))

 
+++++++++ Ricci(1,0) ++++++++++++++++


...

(-8*c**2*k*a(t) - 2*a(t)*Derivative(a(t), (t, 2)) + Derivative(a(t), t)**2)/(4*c**2*(k*r**2 - 1)*a(t))

 
+++++++++ Ricci(2,2) ++++++++++++++++


(2*k*r**2 + a(t) - 1)/a(t)

 
+++++++++ Ricci(3,3) ++++++++++++++++


(2*k*r**2 + a(t) - 1)*sin(th)**2/a(t)